# PySpark 1: foundations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/spark/pyspark-01-foundations.ipynb)

PySpark is the Python API for Apache Spark. This module covers what it is, how a session runs work, the difference between DataFrames and RDDs, and why transformations build a plan that only runs on an action.

**You will learn to:**

- Explain where PySpark fits in a data platform and when not to use it
- Create a session and read a DataFrame's plan with `explain`
- Tell transformations from actions, and predict what executes
- Use `explain` to see filter and column pushdown

**Before you start:** basic Python and SQL. No Spark experience is needed.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

No code in this part: it defines the words every later section uses - driver and executor,
DataFrame and RDD, lazy plan.

## What is PySpark and where does it fit in a data platform?

PySpark is the Python API for Apache Spark. It lets you write distributed data processing jobs in Python while Spark executes the work across a cluster.

In a modern lakehouse, PySpark is often used for:
+ Reading raw files from S3, ADLS, GCS, Kafka, or relational sources
+ Cleaning and standardizing semi-structured data
+ Joining large fact data with dimensions
+ Writing Delta, Iceberg, or Parquet tables for analytics
+ Running batch pipelines, streaming pipelines, and feature engineering jobs

Typical production flow:
```text
raw source files -> PySpark bronze job -> PySpark silver job -> Unity Catalog gold tables -> BI / ML / reverse ETL
```

Real-world example: an orders pipeline lands JSON in S3 every 15 minutes. A PySpark job reads the raw JSON, enforces a schema, deduplicates by `order_id`, joins customer data, and writes a governed table such as `main.gold.fact_order`.

## How do DataFrames differ from RDDs in PySpark?

DataFrames are structured, schema-aware distributed datasets. Spark can optimize DataFrame operations using Catalyst and Tungsten. RDDs are lower-level distributed collections where Spark sees less structure and has fewer optimization opportunities.

How it works under the hood. Catalyst sees DataFrame expressions and rewrites them with rules such as predicate pushdown, column pruning and constant folding; Tungsten then compiles the physical operators into one generated Java method over a compact binary row format. An RDD holds opaque Python or JVM objects, so the optimizer cannot see inside a lambda, and `df.rdd` pays a per-row conversion between JVM rows and Python objects.

Worked example. A `filter` on a DataFrame compiles into a single generated method over binary rows. The same predicate written as `df.rdd.filter(lambda r: ...)` ships Python objects across the Py4J boundary for every row and can be 5-20x slower on a wide table.

Common mistake: `df.rdd.map(...).toDF()` to express logic that `pyspark.sql.functions` already covers. You keep the syntax you know but lose predicate pushdown and column pruning.

## When you would NOT use PySpark

Spark is a heavy default, not a habit. Reach for something simpler when:

+ the data fits in one machine's memory (a few GB) - pandas, DuckDB or plain SQL is faster to write, run and iterate on;
+ the work is pure relational SQL over warehouse tables - the warehouse engine (or dbt) avoids shipping and maintaining Python for what SQL already does;
+ latency-sensitive serving and row lookups are the job - pipelines are batch animals; point lookups belong in a database;
+ the source system can do the transform for you (a streaming engine, a view), because every extra engine multiplies the operational surface.

The honest two-line interview answer: "PySpark earns its cost once data volume, streaming,
or heavy custom Python logic appears; before that, simpler engines win."

Everything above was words; here the machine appears. One idea to carry through this
part: **nothing runs until it must**. Create the session, seed the small dataset the
lessons below share, read with an explicit schema, transform lazily, and only then
pull the trigger with an action.

## How do you create a SparkSession in PySpark?

`SparkSession` is the main entry point for DataFrame, SQL, catalog, and streaming operations. In Databricks notebooks, `spark` is usually already available. In jobs, tests, or local development, you create it explicitly.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

On Databricks with Unity Catalog the session already exists and tables have three-part names:

```python
orders = spark.table("main.silver.orders")
```

This notebook uses a local session catalog instead, so table names below have one or two parts.

How it works under the hood. `SparkSession.builder` reads static configs, then `getOrCreate()` returns the existing session if one is already running in this JVM. Configs passed to the builder are fixed for the life of the session; `spark.conf.set(...)` can change runtime SQL settings later. Every shuffle uses `spark.sql.shuffle.partitions` as its reduce fan-out.

Worked example, the defaults you should know:

```text
spark.sql.shuffle.partitions          200
spark.sql.autoBroadcastJoinThreshold  10485760 bytes (10 MB)
spark.sql.adaptive.enabled            true (Spark 3.2+)
spark.sql.files.maxPartitionBytes     134217728 bytes (128 MB)
```

Common mistake: calling `getOrCreate()` again with different configs and expecting them to apply. It silently returns the first session, so stop the old one or use `spark.conf.set` for runtime settings.

How it works under the hood. A PySpark program has one driver process that owns the `SparkSession`, and one or more executor processes that own the data. The driver turns your DataFrame calls into a logical plan, Catalyst rewrites it, and the DAG scheduler cuts it into stages at every shuffle. Executors run tasks on partitions and report metrics back; `master("local[*]")` packs driver and one executor into a single JVM, which is why it is a development tool and not a deployment topology.

Worked example. A nightly job reads 2 TB of Parquet in 200 files. The driver holds the plan and task metadata (a few MB), while executors hold shuffle files, cached blocks and broadcast tables. With a 2 GB driver and 16 GB executors, the driver limit is the plan plus anything you `collect()`; the executor limit is shuffle, cache and broadcast.

Common mistake: assuming PySpark runs Python on every worker. Python only runs for UDFs and driver-side code; the DataFrame engine runs in the JVM, which is exactly why built-ins are fast and Python UDFs are not.

## Sample data: the tables every lesson below runs on

This section seeds a small four-table slice of an orders platform, built with
`spark.createDataFrame` and an explicit schema per table. Three quirks the lessons lean on:
customer 4 has no `region`, customers 7 and 8 have never ordered, and one event is an exact
retry of the same row.

| table | grain | notable columns |
| --- | --- | --- |
| `customers` | one row per customer | `region`, `signup_date` (`DATE`) |
| `orders` | one row per order | `status`, `order_ts` (`TIMESTAMP`), `amount` (decimal) |
| `order_items` | one row per order line | `qty`, `unit_price` |
| `events` | one row per user event | `user_id`, `event`, `ts` |

### `customers` (8 rows)

Customer 4's `region` never arrived from the CRM - the null-handling cell below fills it.

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

### `orders` (12 rows)

`order_ts` is a `TIMESTAMP` and `amount` a decimal, and every order's lines in the next table add
up to its amount exactly. Orders 2 and 3 belong to one customer on the same day, and order 12
lands just after midnight UTC.

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

### `order_items` (17 rows)

SKUs repeat across orders, so `countDistinct("order_id")` has real work to do, and order 9 is a
bulk reorder of twenty units of `SKU-4011`.

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

### `events` (13 rows, one is a retry)

One of user 5's `view` events was committed twice by a retried SDK call - the exact shape the
dedupe cell below collapses.

In [ ]:
from pyspark.sql import types as T

events_schema = T.StructType([
    T.StructField("user_id", T.IntegerType(), False),
    T.StructField("event", T.StringType(), False),
    T.StructField("ts", T.TimestampType(), False),
])

events = spark.createDataFrame(
    [
        (1, "view",        datetime(2026, 1,  6,  9,  0)),
        (1, "add_to_cart", datetime(2026, 1,  6,  9,  2)),
        (1, "purchase",    datetime(2026, 1,  6,  9, 10)),
        (2, "view",        datetime(2026, 1,  7, 10,  0)),
        (2, "purchase",    datetime(2026, 1,  7, 10, 40)),
        (3, "view",        datetime(2026, 1,  6, 11, 15)),
        (3, "view",        datetime(2026, 1,  8, 11, 20)),
        (4, "add_to_cart", datetime(2026, 1,  9, 20,  5)),
        (4, "purchase",    datetime(2026, 1, 10,  8, 30)),
        (5, "view",        datetime(2026, 1, 11, 14, 45)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
        (5, "purchase",    datetime(2026, 1, 12, 12, 30)),
        (5, "view",        datetime(2026, 1, 12,  9,  0)),
    ],
    schema=events_schema,
)

events.orderBy("user_id", "ts").show()

### Row-count sanity check

Four exact counts plus the two invariants the lessons depend on: every order's lines reconcile
with its amount, and the retried event collapses to one row.

In [ ]:
from pyspark.sql import functions as F

frames = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "events": events,
}
expected_rows = {"customers": 8, "orders": 12, "order_items": 17, "events": 13}

for table, frame in frames.items():
    actual = frame.count()
    assert actual == expected_rows[table], (table, actual, expected_rows[table])
    print(f"{table:<12} {actual:>2} rows")

per_order = (
    order_items
    .groupBy("order_id")
    .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("lines"))
    .join(orders.select("order_id", "amount"), "order_id")
)
assert per_order.where(F.col("lines") != F.col("amount")).count() == 0
print("line totals reconcile with the orders ledger:", float(orders.agg(F.sum("amount")).first()[0]))

assert events.dropDuplicates().count() == events.count() - 1
print("and the retried event collapses to 12 distinct rows - seed verified")

### DataFrames versus RDDs, in code

Prefer DataFrames for data engineering work - Catalyst can see every column expression:

In [ ]:
from pyspark.sql import functions as F

paid_orders = (
    orders
    .where(F.col("status") == "paid")
    .select("order_id", "customer_id", "amount", "order_ts")
)
paid_orders.orderBy("order_id").show()
assert [r["order_id"] for r in paid_orders.orderBy("order_id").collect()] == [1, 4, 7, 10]

Use RDDs only for record-level logic the DataFrame API cannot express. The lambda below is
opaque to the optimizer and runs in Python workers:

In [ ]:
log_lines = ["GET /orders 200", "POST /orders 201", "GET /health 200"]
line_lengths = spark.sparkContext.parallelize(log_lines).map(len)

print(line_lengths.collect())
assert line_lengths.collect() == [15, 16, 15]

## What are transformations and actions in PySpark?

Transformations build a logical plan. Actions trigger execution.

Transformations only build a plan:

In [ ]:
daily_revenue = (
    orders
    .where(F.col("status").isin("paid", "shipped"))
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date")
    .agg(F.sum("amount").alias("revenue"))
)
print(type(daily_revenue).__name__, "- nothing has executed yet")

An action triggers execution. Each of these runs the whole lineage again:

In [ ]:
print("days with revenue:", daily_revenue.count())
daily_revenue.write.mode("overwrite").saveAsTable("daily_revenue")

total = spark.table("daily_revenue").agg(F.sum("revenue")).first()[0]
print("revenue written:", total)
assert float(total) == 2273.90

Common transformations include `select`, `where`, `withColumn`, `join`, `groupBy`, `agg`, `distinct`, and `dropDuplicates`. Common actions include `count`, `collect`, `show`, `take`, `write`, and `foreachBatch`.

How it works under the hood. Transformations only add nodes to the logical plan; nothing is scheduled. An action triggers Catalyst optimization, physical planning, a DAG split into stages at each shuffle, and one task per partition. That is why the same DataFrame walked twice by two actions is computed twice unless you cached it.

Worked example. A `groupBy("order_date").agg(sum)` over 500 GB shows `HashAggregate(partial) -> Exchange -> HashAggregate(final)`. The partial aggregate runs inside each map task and shrinks the rows before the network, so the shuffle moves far less than 500 GB; that two-node pattern is the signature of a decomposable aggregate.

Common mistake: calling `count()` or `show()` between every step to "check progress". Each call re-runs the whole lineage from the source; cache the frame or read `explain("formatted")` instead.

## How does lazy evaluation work in PySpark?

Lazy evaluation means Spark waits until an action before executing the plan. This lets Spark optimize the whole plan instead of executing every line immediately.

To see pushdown, write `orders` as Parquet and read it back - `explain` then shows what reaches
the file reader. The helper captures the plan text so a cell can assert on it.

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()


orders_path = os.path.join(WORK_DIR, "orders_parquet")
orders.write.mode("overwrite").parquet(orders_path)

result = (
    spark.read.parquet(orders_path)
    .where("order_ts >= timestamp '2026-01-10'")
    .select("order_id", "customer_id", "amount")
    .where("amount > 0")
)
plan = plan_of(result)
print(plan)
read_schema = next(line for line in plan.splitlines() if line.startswith("ReadSchema"))
assert "GreaterThan(amount,0.00)" in plan  # the filter reached the Parquet reader
assert "status" not in read_schema  # unused columns are never read

Spark can combine filters, prune unused columns, push predicates to Parquet/Delta/Iceberg readers, and choose join strategies before running tasks.

Real-world example: if you read a 2 TB orders table but select only `order_id` and `amount`, Spark can avoid reading unused columns from Parquet files. That optimization is possible because the plan is declarative.

How it works under the hood. `explain("formatted")` prints the analyzed and optimized physical plan without reading data. Catalyst applies rule-based passes (predicate pushdown, constant folding, column pruning, null propagation) and the planner chooses join and aggregate strategies from table statistics. With AQE on, the printed plan is provisional and can change at runtime.

Worked example. `.where(ts >= ...).select(a, b).where(a > 0)` collapses into one `FileScan parquet` node with `PushedFilters: [IsNotNull(a), GreaterThan(a,0)]` and `ReadSchema: a,b`. Two filters and a projection became a single scan; no extra stage was created.

Common mistake: trusting the initial plan as final. AQE plans print `isFinalPlan=false` until an action runs, and `explain("cost")` shows the row estimates the planner used.

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- A DataFrame is a plan. Nothing runs until an action such as `count`, `show` or a write.
- Prefer DataFrame functions to RDDs. Catalyst can optimize them; Python lambdas hide the logic from it.
- Read `explain` output before you tune anything.

**Next:** [PySpark 2: reading, SQL and cleaning](pyspark-02-reading-sql-and-cleaning.ipynb)